In [ ]:
# -*- coding: utf-8 -*-

def reorganizar_datos(archivo_entrada, archivo_salida):
    """
    Lee un archivo de texto con datos en múltiples columnas y los escribe
    en un nuevo archivo en una sola columna.

    Args:
        archivo_entrada (str): La ruta del archivo con los datos originales.
        archivo_salida (str): La ruta del archivo donde se guardará el resultado.
    """
    print(f"Leyendo datos desde '{archivo_entrada}'...")

    try:
        # Abrimos el archivo de salida en modo escritura ('w')
        with open(archivo_salida, 'w') as f_salida:
            # Abrimos el archivo de entrada en modo lectura ('r')
            with open(archivo_entrada, 'r') as f_entrada:
                # Leemos todas las líneas del archivo
                for linea in f_entrada:
                    # Dividimos la línea en valores individuales usando el espacio/tab como separador
                    valores = linea.split()

                    # Escribimos cada valor en el archivo de salida, seguido de un salto de línea
                    for valor in valores:
                        f_salida.write(valor + '\n')

        print(f"¡Proceso completado! Los datos han sido guardados en '{archivo_salida}'.")

    except FileNotFoundError:
        print(f"Error: No se pudo encontrar el archivo '{archivo_entrada}'.")
    except Exception as e:
        print(f"Ocurrió un error inesperado: {e}")

# --- Configuración ---
# Nombre del archivo que contiene tus datos de 8 columnas
nombre_archivo_original = 'Datos_preprocesados.txt'

# Nombre del archivo que se creará con los datos en una sola columna
nombre_archivo_final = 'Datos_en_1_columna.txt'
# -------------------

# Llamamos a la función para que realice el trabajo
reorganizar_datos(nombre_archivo_original, nombre_archivo_final)

In [ ]:
# -*- coding: utf-8 -*-

import numpy as np

def generar_eje_de_tiempo(numero_de_puntos, tiempo_total_us, archivo_salida):
    """
    Genera una secuencia de tiempo linealmente espaciada y la guarda en un archivo.

    Args:
        numero_de_puntos (int): El número total de puntos de datos (ej: 8008).
        tiempo_total_us (float): La duración total de la ventana de tiempo en microsegundos (µs).
        archivo_salida (str): La ruta del archivo donde se guardará el eje de tiempo.
    """
    print(f"Generando eje de tiempo con {numero_de_puntos} puntos...")

    try:
        # Usamos np.linspace para crear un arreglo de N puntos igualmente espaciados
        # que van desde 0 hasta el tiempo_total_us.
        eje_tiempo = np.linspace(start=0, stop=tiempo_total_us, num=numero_de_puntos)

        # Abrimos el archivo de salida y guardamos cada punto de tiempo.
        # Usamos un formato con 6 decimales para una buena precisión.
        with open(archivo_salida, 'w') as f:
            for tiempo in eje_tiempo:
                f.write(f"{tiempo:.6f}\n")

        print(f"¡Proceso completado! El eje de tiempo ha sido guardado en '{archivo_salida}'.")

    except Exception as e:
        print(f"Ocurrió un error inesperado: {e}")

# --- Configuración ---
# El número de puntos debe coincidir con tus datos de la señal.
# En el procesamiento anterior eran 8004, pero lo ajusto a 8008 como pides.
PUNTOS_TOTALES = 8008

# Tiempo total de la captura en microsegundos (250 µs/div * 10 div)
TIEMPO_TOTAL_MICROSEGUNDOS = 8008.0

# Nombre del archivo que se creará con los datos del tiempo
NOMBRE_ARCHIVO_TIEMPO = 'Datos_eje_tiempo.txt'
# -------------------

# Llamamos a la función para que realice el trabajo
generar_eje_de_tiempo(PUNTOS_TOTALES, TIEMPO_TOTAL_MICROSEGUNDOS, NOMBRE_ARCHIVO_TIEMPO)

In [ ]:
# -*- coding: utf-8 -*-

import matplotlib.pyplot as plt
import numpy as np

def graficar_senal_vs_tiempo(archivo_tiempo, archivo_senal):
    """
    Lee un archivo con datos de tiempo y otro con datos de la señal,
    y los grafica uno contra el otro.

    Args:
        archivo_tiempo (str): Ruta del archivo con el eje de tiempo (X).
        archivo_senal (str): Ruta del archivo con los valores de la señal (Y).
    """
    eje_x_tiempo = []
    eje_y_valores = []

    print(f"Leyendo eje de tiempo desde '{archivo_tiempo}'...")
    print(f"Leyendo datos de la señal desde '{archivo_senal}'...")

    try:
        # --- Leer el archivo de tiempo (Eje X) ---
        with open(archivo_tiempo, 'r') as f_tiempo:
            for linea in f_tiempo:
                eje_x_tiempo.append(float(linea.strip()))

        # --- Leer el archivo de la señal (Eje Y) ---
        with open(archivo_senal, 'r') as f_senal:
            for linea in f_senal:
                valor_texto = linea.strip()
                if valor_texto:
                    valor_texto_punto = valor_texto.replace(',', '.')
                    eje_y_valores.append(float(valor_texto_punto))

        # Verificación para asegurar que ambos archivos tienen la misma cantidad de datos
        if len(eje_x_tiempo) != len(eje_y_valores):
            print("¡Advertencia! Los archivos de tiempo y señal no tienen la misma cantidad de puntos.")
            print(f"Puntos de tiempo: {len(eje_x_tiempo)}, Puntos de señal: {len(eje_y_valores)}")
            # Se podría detener aquí o intentar graficar hasta el mínimo de puntos
            min_puntos = min(len(eje_x_tiempo), len(eje_y_valores))
            eje_x_tiempo = eje_x_tiempo[:min_puntos]
            eje_y_valores = eje_y_valores[:min_puntos]

        # --- Creación de la Gráfica ---
        plt.figure(figsize=(12, 6))

        # La principal diferencia: ahora plt.plot() recibe los datos de X y de Y
        plt.plot(eje_x_tiempo, eje_y_valores, linestyle='-')

        plt.title('Señal digitalizada por el osciloscopio')
        plt.xlabel('Tiempo (µs)')
        plt.ylabel('Tensión (V)')
        plt.grid(True)
        
        # Opcional: ajustar límites del eje X si es necesario
        plt.xlim(min(eje_x_tiempo), max(eje_x_tiempo))
        plt.ylim(0, 22)
        plt.yticks(np.arange(0, 22, 2))
        
        plt.tight_layout()
        
        nombre_grafica = 'Grafico.png'
        plt.savefig(nombre_grafica)
        print(f"La gráfica ha sido guardada como '{nombre_grafica}'")

        plt.show()

    except FileNotFoundError as e:
        print(f"Error: No se pudo encontrar uno de los archivos. Detalles: {e}")
    except Exception as e:
        print(f"Ocurrió un error inesperado: {e}")


# --- Configuración ---
nombre_archivo_tiempo = 'Datos_eje_tiempo.txt'
nombre_archivo_senal = 'Datos_en_1_columna.txt'
# -------------------

# Llamamos a la función para que grafique ambos conjuntos de datos
graficar_senal_vs_tiempo(nombre_archivo_tiempo, nombre_archivo_senal)